# Huấn luyện Deep RL trên Kaggle

Thay URL bằng repo công khai và hoàn thiện `ai.deep_rl.user.train` trước khi chạy.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/OWNER/REPOSITORY.git"
repo = Path("/kaggle/working/chess-ai")
subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "chess"], check=True)

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "GPU not detected")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "ai.deep_rl.user.train", "--out", "/kaggle/working/run1"],
    cwd=repo,
    check=True,
)

In [ ]:
checkpoint = Path("/kaggle/working/run1/best.pt")
if checkpoint.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ai.deep_rl.evaluate",
            "--weights",
            str(checkpoint),
            "--opponent",
            "random",
            "--games",
            "50",
        ],
        cwd=repo,
        check=True,
    )
else:
    print("Training output did not create best.pt; set checkpoint to its saved policy file.")

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from ai.deep_rl.metrics import plot

run = Path("/kaggle/working/run1")
metrics_csv = run / "metrics.csv"
if metrics_csv.exists():
    plot(metrics_csv, run / "metrics.png")
with ZipFile("/kaggle/working/run1.zip", "w", ZIP_DEFLATED) as archive:
    for path in run.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(run.parent))